In [ ]:
import os
import sys

from corner import corner
from copy import deepcopy 

sys.path.append('..')

from src.simulator import Model, BurstSimulator
from src.flow_matching.simulator import Model as BatchedModel

from src.c2st import c2st
from src.flow_matching.loader import empty_classifier_from_config, empty_model_from_config, read_config, prob_path_from_config
from src.flow_matching.distributions import UniformPrior, CompositePrior, Posterior, DiscreteUniform, StandardGaussian
from src.flow_matching.probability_path import GuidedLinearProbabilityPath
from src.flow_matching.integration import EulerODESolver
from src.flow_matching.models import MLPGuidedVectorField, FRBLightCurveCNN, LightCurveThinner, fourier_embedding, LightCurveMLP, UNetEncoder, TransdimensionalModel, EncodedClassifier
from src.flow_matching.transformer import TransformerGuidedField
from src.helpers import record_every, plot_posterior_samples, gen_parameter_labels

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import matplotlib.lines as mlines

import numpy as np
import torch
import yaml

from src.flow_matching.plotting import plot_loss, plot_snapshots
from src.flow_matching.helpers import choose_device, build_mlp, find_run_dir

device = choose_device()

# Loading in the FM model

In [ ]:
# fill in desired job_id or directory name 
job_id = "15027662" #False
run_dir = None
save_dir = "../checkpoints/"

In [ ]:
# loading the model (via run_id, or path)
run_dir = find_run_dir(job_id, save_dir) if job_id else os.path.join(save_dir, run_dir)

checkpoint_path = os.path.join(run_dir, 'training_checkpoint.pth')
config_path     = os.path.join(run_dir, 'config.yaml')

In [ ]:
# create empty model from config
config = read_config(config_path)
vector_field = empty_model_from_config(config)

In [ ]:
transdimensional = not config['training']['fixed_N'] #False
print(transdimensional)
if transdimensional:
    classifier = empty_classifier_from_config(config)
    vector_field = TransdimensionalModel(classifier, vector_field)

In [ ]:
# load trained model 
print(checkpoint_path)
checkpoint = torch.load(checkpoint_path, weights_only=False)

# Load 'normal' or EMA version 
ema = True
if ema:
    EMA_checkpoint_path = os.path.join(run_dir, "EMA_checkpoint.pth")
    state_dict = torch.load(EMA_checkpoint_path, weights_only=False)
    vector_field.load_state_dict(state_dict) 
else:
    vector_field.load_state_dict(checkpoint["model_state_dict"])

vector_field.eval()
vector_field.to(device)

losses = checkpoint["losses"]

In [ ]:
path = prob_path_from_config(config)
inf_params = config['model']['init_params']['inf_params']
N = path.p_data.model_params['ncomp']
vector_dim = N * len(inf_params)
burstparams = path.p_data.model_params['burstparams']

try:
    mean, std = torch.tensor(config['training']['sample_mean'], device=device), torch.tensor(config['training']['sample_std'], device=device)
except KeyError:
    mean, std = torch.zeros(vector_dim, device=device), torch.ones(vector_dim, device=device)

path.p_data.N_prior.device = device
path.p_data.device = device
prior = path.p_data.prior
# path.p_simple.device = device
# path.p_simple.set_prior_device()

# generate FM posterior conditioned on simulated counts

In [ ]:
# sample lightcurve to condition on
print(path.p_data.prior)

params, simulated_counts, components = path.p_data.sample(1)
burstparams = prior.samples_as_dict(params)
N_true = components[0][0].item()
print("This example has N = ", components[0][0].item(), " components")
print(
    f"""
    The burst parameters are:
    t0 =\t {np.array_str(burstparams['t0'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(amp) =\t {np.array_str(burstparams['amp'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    log(rise) =\t {np.array_str(burstparams['rise'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    skew =\t {np.array_str(burstparams['skew'][0, :components].cpu().numpy(), precision=2, suppress_small=True)}
    """
)
simulated_counts = simulated_counts[0]
plt.plot(simulated_counts.cpu().numpy())
plt.show()

In [ ]:
condition = torch.tensor(simulated_counts / 150, device=device, dtype=torch.float)
logits = classifier(condition.unsqueeze(0))
p_N = torch.softmax(logits, dim=1).flatten()
plt.plot(p_N.detach().cpu())
plt.show()

In [ ]:
# path.p_simple.priors['t0'].device = device 

num_samples = 25000  # number of prior samples to transform 
samples_per_batch = 5000
batches = num_samples // samples_per_batch
final_snapshot = torch.zeros((batches * samples_per_batch, vector_dim), device=device)
Ns = torch.zeros((batches * samples_per_batch, 1), device=device)

# use same data point for conditioning all prior samples
condition = torch.tensor(simulated_counts / 150, device=device, dtype=torch.float)
simulations = condition.repeat(samples_per_batch, 1)

# initialize ODE solver
solver = EulerODESolver(vector_field)
nts = 200
ts = torch.linspace(0, 1, nts).to(device)

if transdimensional:
    # logits = classifier(condition.unsqueeze(0))
    logits = classifier(condition.unsqueeze(0))
    p_N = torch.softmax(logits, dim=1).flatten()
else:
    p_N = torch.zeros(N, device=device)
    p_N[N-1] = 1
    
# integrate in batches
for i in range(batches):
    # simulate ODE starting from x0

    start = i * samples_per_batch
    stop = start + samples_per_batch

    N_samples = torch.multinomial(p_N.repeat(samples_per_batch, 1), num_samples=1) + 1

    x0 = path.p_simple.sample(samples_per_batch, Ns=N_samples).to(device) 

    Ns[start:stop, :] = N_samples
    final_snapshot[start:stop, :] = solver.solve(x0, ts.view(1, nts, 1).expand(samples_per_batch, nts, 1), y=simulations, N=N_samples) * std + mean

In [ ]:
final_snapshot

# sorting in case of uniform prior

In [ ]:
sorted_snapshot = final_snapshot.reshape(-1, len(inf_params), N).clone()

# sort param columns based on peaktime row
peaktime_idx = inf_params.index('t0')

# NOTE : IF TRANSDIMENSIONAL, HAVE TO CUT OFF / MASK IRRELEVANT TOKENS FIRST, THEN SORT.
if transdimensional:
    mask = torch.arange(1, N+1, device=Ns.device).unsqueeze(0) > Ns  # (5000, N)
    print(mask, Ns)
    sorted_snapshot[:, peaktime_idx, :][mask] = sys.maxsize

indeces = sorted_snapshot[:, peaktime_idx, :].argsort(dim=-1)
indeces_expanded = indeces.unsqueeze(1).expand(-1, len(inf_params), -1)

sorted_snapshot = torch.gather(sorted_snapshot, dim=-1, index=indeces_expanded)

# undo max size for stability
if transdimensional:
    sorted_snapshot[:, peaktime_idx, :][mask] = 0
    
sorted_snapshot = sorted_snapshot.view(-1, vector_dim)
sorted_snapshot

In [ ]:
# plot sorted peaktimes
plt.figure(figsize=(20, 3))
indeces = np.random.choice(range(num_samples), 200)
plt.plot(sorted_snapshot[indeces][:, :1].detach().cpu().numpy(), 'o')
plt.plot(sorted_snapshot[indeces][:, 1:2].detach().cpu().numpy(), 'o')
plt.plot(sorted_snapshot[indeces][:, 2:3].detach().cpu().numpy(), 'o')

# Classifier p(N)

In [ ]:
plt.bar(range(1, len(p_N.cpu().detach().numpy())+1), height=p_N.cpu().detach().numpy())
plt.xlabel("$N_{pred}$")
plt.title('p(N|y)')

print("True number of components is: ", components[0][0].item())
print("Most likely estimate from classifier: ", range(1, len(p_N.cpu().detach().numpy())+1)[torch.where(p_N == torch.max(p_N))[0]])

# corner plot (for a given N)

In [ ]:
range_ = 0.95
N_inf = 3 # choose for which N to make corner plot (for FM)
N_inf = N if not transdimensional else N_inf 
FM_samples = sorted_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf
# FM_samples = final_snapshot[(Ns == N_inf).view(Ns.size(0))]# samples where N == N_inf
print(FM_samples.shape)
print(final_snapshot.shape)
# make vector correct size 
# (f.e. if N=2 select t0_1, t0_2, rise_1, rise_2 from vector structured like [t0_1 ... t0_Nmax,  rise_1 ... rise_Nmax])
bs, dim = FM_samples.shape
new_dim = N_inf * len(inf_params)
print( FM_samples.view(bs, len(inf_params), N)[:, :, :N_inf].shape)
FM_samples = FM_samples.view(-1, len(inf_params), N)[:, :, :N_inf].reshape(bs, new_dim)  

var_names = gen_parameter_labels(inf_params, N)
var_names = np.array(var_names).reshape(len(inf_params), N)[:, :N_inf].flatten()
true_values = np.array([burstparams[key][0][:N_inf] for key in inf_params]).flatten()
print(true_values)
true_values = true_values.reshape(len(inf_params), N_inf)
indeces = true_values[peaktime_idx, :].argsort(axis=-1)
true_values = true_values[:, indeces].flatten()
print(true_values)
fig = corner(FM_samples.cpu().numpy(), labels=var_names, truths=true_values, range=[range_ for _ in range(N_inf * len(inf_params))], 
            color='black', label='FM', plot_density=True, plot_datapoints=True, 
            fill_contours=False, plot_contours=True, hist_kwargs={'density':True}, bins=20)

plt.show()


## posterior samples

In [ ]:
# posterior samples that include all sampled component numbers
modelparams = {'time':torch.linspace(0, 1, simulated_counts.shape[0]), 'burstparams': burstparams, 'ybkg':5,'ncomp':components}

true_flux = BatchedModel(**modelparams, device=device).get_flux().flatten()

plt.figure(figsize=(20,5))
plt.subplot(121)
plt.plot(simulated_counts.flatten().cpu(),'k-',alpha=0.5,  label='noisy flux')

posterior_param_samples = prior.samples_as_dict(final_snapshot)
# Ns_10 = torch.ones_like(Ns) * 2
posterior_curve_samples = BatchedModel(
    device=device, 
    **{
        'time':torch.linspace(0, 1, simulated_counts.shape[0]), 
        'burstparams':posterior_param_samples,
        'ybkg':5,
        'ncomp':Ns#_10
        }
        ).get_flux()

for i in range(10000, 20000):
    random_idx = torch.randint(0, num_samples, size=(1,)).item()
    plt.plot(posterior_curve_samples[random_idx].cpu(), alpha=0.75)
plt.title('100 posterior samples')
plt.subplot(122)
plt.plot(simulated_counts.flatten().cpu(),alpha=0.5, label='poisson flux')
# plt.plot(true_flux.cpu(), 'k--',label='ground truth')
plt.legend()

plt.show()
print(simulated_counts.shape)

In [ ]:
if transdimensional:
    MSE_loss = checkpoint['MSE_loss']
    CEL_loss = checkpoint['CEL_loss']
    plt.loglog(MSE_loss, label='MSE')
    plt.loglog(CEL_loss, label='CEL')
    plt.legend()

# p(N) for collection of lightcurves

In [ ]:
examples = 2000
path.p_data.device = device
path = path.to(device)

for key in prior.priors:
        prior.priors[key].device=device
path.p_data.N_prior.device=device

plt.figure(figsize=(15, 3))
for i, n in enumerate([1, 3, 5, 8]):
    plt.subplot(1,4,i+1)
    path.p_data.N_prior = DiscreteUniform(n,n, device=device)
    
    path = path.to(device)
    params, counts, components = path.p_data.sample(examples)
    burstparams = prior.samples_as_dict(params)

    model = BatchedModel(
                time=torch.linspace(0, 1, 1000), 
                ncomp=components, 
                ybkg=5.0, 
                burstparams=burstparams,
                device=device
                )
    true_flux = model.get_flux()
    condition = torch.tensor(counts / 150, device=device, dtype=torch.float)

    # initialize ODE solver
    solver = EulerODESolver(vector_field)
    nts = 200
    ts = torch.linspace(0, 1, nts).to(device)

    logits = classifier(condition)
    p_N = torch.softmax(logits, dim=1)
    colors = ['black' for i in range(10)]
    colors[n-1]='tab:green'
    # plt.grid(axis='y', zorder=-1)
    plt.bar(range(1,11), torch.mean(p_N, axis=0).flatten().cpu().detach(), color=colors, alpha=0.6)
    # plt.title(f'average p(N|y) for {examples} sample curves with N={n} components')
    plt.title(f'$N_{{true}}$ = {n}')#, x=0.85, y=0.9)
    plt.ylabel('⟨p(N)⟩', fontsize=14) if i == 0 else None
    plt.xlabel('N', fontsize=14) 
    plt.yticks(ticks=[0.2, 0.4, 0.6, 0.8, 1.0], labels=([] if i > 0 else None), fontsize=12)
    plt.xticks(fontsize=12)
    plt.ylim(0,1)
    
# plt.tight_layout()
plt.show()

In [ ]:
# lightcurve vs classifier result
# for i in range(examples):
    #     plt.figure(figsize=(15, 5))
    #     plt.subplot(121)
    #     plt.bar(range(1, 11), height=p_N[i].flatten().cpu().detach().numpy())
    #     plt.xlabel("$N_{pred}$")
    #     plt.title('p(N|y)')

    #     N_true = components[i].item()
    #     true_parameters = path.p_simple.samples_as_dict(params[i][:N_true].view(1,-1))

    #     plt.subplot(122)
    #     plt.title(f'N_true = {N_true}')
    #     plt.plot(torch.linspace(0, 1, 1000), counts[i].cpu(), 'k-', label='simulated flux')
    #     plt.vlines(true_parameters['t0'].flatten(), ymin= torch.zeros_like(true_parameters['t0'].flatten()), ymax=torch.max(counts[i]).repeat(N_true).cpu(), linestyle='dotted', label='t_0')
    #     plt.plot(torch.linspace(0, 1, 1000), true_flux[i].cpu(), 'r-', alpha=0.7, label='ground truth')
    #     plt.legend(bbox_to_anchor=(1.3, 1.02))

    #     print(true_parameters)